# HomeHand — flat-box grasp with PPO on MJX

Settings (right panel): **Accelerator: GPU T4 x2**, **Internet: on**, **Persistence: Files only**. No dataset needed:
the environment, the trainer and the MuJoCo model are written out by the cells below.

Run with **Save Version → Save & Run All (Commit)**. Everything in `/kaggle/working/out` (policy, progress) becomes
the version's output; `flat_grasp_out.zip` holds all of it.

In [ ]:
!pip install -q "jax[cuda12]==0.9.2" "flax==0.12.6" "brax==0.14.2" "mujoco==3.15.0" "mujoco-mjx==3.15.0"
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
os.environ["XLA_FLAGS"] = os.environ.get("XLA_FLAGS", "") + " --xla_gpu_triton_gemm_any=True"
import jax; print(jax.__version__, jax.devices())

In [ ]:
%%writefile flat_grasp_env.py
"""Flat-box grasping: the RL environment (MJX, for training on a GPU) and its plain-MuJoCo twin (evaluation, the
expert baseline, videos).

The arm starts in the expert's pre-grasp pose (`flat_grasp_model`). Each episode draws:

* the box: +-3 cm around the nominal spot, yaw +-0.6 rad;
* what the camera tells about it: a constant per-episode bias (6 mm in x / y, 3 mm in z, 4 deg in yaw: the size of
  the pose errors of the conveyor / tidy perception), plus 2 mm of noise per frame;
* physics: grip friction (the finger pads) 0.6-1.5, box mass 80-300 g.

Both controllers see the same thing: the joints, the palm and fingertips, the observed box pose, and which fingers
touch the box (the hand's force sensors). The expert plans once from the observed pose (as `skills.top_grasp_plan`):
turn, come down, close, lift - joint targets from the arm IK. The policy acts at 50 Hz on the same observation, so it
can feel where the box really is and how it moves in the hand.

Action (13, in [-1, 1]): rate of change of the position targets of the 7 right-arm joints and the 6 hand actuators.
Success: at the end of the 5 s episode the box is >= 6 cm above the counter, held by the thumb and a finger.

The observation / reward code below takes the array module (`np` or `jax.numpy`) so both twins share it; it only
needs `flat_grasp_model`'s two generated files (no `homehand` import: it is copied into the Kaggle notebook).
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

import mujoco
import numpy as np

DT = 0.02                          # policy at 50 Hz
EPISODE_STEPS = 250                # 5 s
TABLE_Z = 0.80
BOX_XY_RANGE, BOX_YAW_RANGE = 0.03, 0.6
BIAS_XY, BIAS_Z, BIAS_YAW, NOISE = 0.006, 0.003, 0.07, 0.002
GRIP_FRICTION = (0.6, 1.5)
BOX_MASS = (0.08, 0.30)
LIFT_OK = 0.06
FINGER_OPEN = np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0])       # thumb turned in, as the expert's pre-grasp shape
FINGER_CLOSE = np.array([1.0, 0.6, 1.47, 1.47, 1.47, 1.47])  # grasps.py: lying_block_pinch
N_ARM = 7
RATE = np.array([1.5] * N_ARM + [4.0] * 6)                   # target change per second at |action| = 1
PARTS = ("thumb", "index", "middle", "ring", "pinky")
OBS_SIZE = 73


@dataclass
class Layout:
    """Indices into the model, and the pre-grasp the episode starts from."""
    act_qadr: np.ndarray
    act_dadr: np.ndarray
    lo: np.ndarray
    hi: np.ndarray
    box_qadr: int
    box_dadr: int
    box_body: int
    box_geom: int
    pad_geoms: np.ndarray         # the finger pads (contact priority 1: theirs is the grip friction)
    part_of_geom: np.ndarray      # geom -> 0..4 finger (thumb first), 5 palm, -1 other
    palm_site: int
    tip_sites: np.ndarray
    qpos0: np.ndarray
    ctrl0: np.ndarray
    box_xy: np.ndarray
    z_rest: float
    palm_pos: np.ndarray          # the pre-grasp palm pose (the expert plans from it)
    palm_quat: np.ndarray
    heading_axis: int             # the palm axis that lies flattest at the pre-grasp: the hand's heading
    heading0: float

    @classmethod
    def of(cls, m: mujoco.MjModel, frame_json: str | Path) -> "Layout":
        frame = json.loads(Path(frame_json).read_text())
        joints = m.actuator_trnid[:, 0]
        part = -np.ones(m.ngeom, dtype=np.int32)
        for g in range(m.ngeom):
            if not (m.geom_contype[g] or m.geom_conaffinity[g]):
                continue
            body = m.body(m.geom_bodyid[g]).name
            for k, name in enumerate(PARTS):
                if body.startswith(f"R_{name}_"):
                    part[g] = k
            if body == "R_hand_base_link":
                part[g] = 5
        pads = np.flatnonzero((part >= 0) & (m.geom_priority > 0)).astype(np.int32)
        qadr = m.jnt_qposadr[joints].astype(np.int32)
        qpos0 = m.qpos0.copy()
        ctrl0 = np.r_[frame["arm_q"], FINGER_OPEN]
        qpos0[qadr] = ctrl0
        d = mujoco.MjData(m)
        d.qpos[:] = qpos0
        mujoco.mj_kinematics(m, d)
        palm = m.site("R_palm").id
        R = d.site_xmat[palm].reshape(3, 3)
        axis = int(np.argmin(np.abs(R[2, :])))
        return cls(
            act_qadr=qadr, act_dadr=m.jnt_dofadr[joints].astype(np.int32),
            lo=m.actuator_ctrlrange[:, 0].copy(), hi=m.actuator_ctrlrange[:, 1].copy(),
            box_qadr=int(m.jnt_qposadr[m.joint("box_free").id]), box_dadr=int(m.jnt_dofadr[m.joint("box_free").id]),
            box_body=m.body("box").id,
            box_geom=m.geom("box").id, pad_geoms=pads, part_of_geom=part, palm_site=palm,
            tip_sites=np.array([m.site(f"R_{p}_tip").id for p in PARTS], dtype=np.int32),
            qpos0=qpos0, ctrl0=ctrl0, box_xy=np.array(frame["box_xy"]), z_rest=TABLE_Z + frame["box_half"][2],
            palm_pos=np.array(frame["palm_pos"]), palm_quat=np.array(frame["palm_quat"]),
            heading_axis=axis, heading0=float(np.arctan2(R[1, axis], R[0, axis])))


# ---------------------------------------------------------------------------------------- shared (np / jnp)
def box_yaw(xp, quat):
    w, x, y, z = quat[0], quat[1], quat[2], quat[3]
    return xp.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))


def observe(xp, lay: Layout, qpos, qvel, site_xpos, site_xmat, ctrl, touch, box_obs, yaw_obs, t_frac):
    """Policy input (73): positions / speeds / targets of the 13 actuated joints, the observed box from the palm and
    from each fingertip, the palm's orientation, the box's heading against the hand's, its observed height, touch,
    time."""
    mid, half = (lay.hi + lay.lo) / 2, (lay.hi - lay.lo) / 2
    q = qpos[lay.act_qadr]
    palm = site_xpos[lay.palm_site]
    tips = site_xpos[lay.tip_sites]
    R = site_xmat[lay.palm_site].reshape(3, 3)
    a = lay.heading_axis
    heading = xp.arctan2(R[1, a], R[0, a]) - lay.heading0
    dyaw = 2 * (yaw_obs - heading)                    # the box looks the same turned by half a turn
    return xp.concatenate([
        (q - mid) / half, 0.1 * qvel[lay.act_dadr], (ctrl - mid) / half,
        10 * (box_obs - palm), 10 * (box_obs[None, :] - tips).reshape(-1), R[:, 0], R[:, 2],
        xp.array([xp.sin(dyaw), xp.cos(dyaw), 10 * (box_obs[2] - lay.z_rest)]), touch, xp.array([t_frac])])


def reward(xp, lay: Layout, qpos, site_xpos, touch, action, box_xy0):
    """Shaped: reach (palm and fingertips to the box), grip (thumb + fingers on it), lift while gripped, success;
    small costs for pushing the box away on the counter and for jerky actions."""
    box = qpos[lay.box_qadr:lay.box_qadr + 3]
    d_palm = xp.linalg.norm(box - site_xpos[lay.palm_site])
    d_tips = xp.linalg.norm(box[None, :] - site_xpos[lay.tip_sites], axis=-1).mean()
    thumb, fingers = touch[0], touch[1:5].sum()
    held = thumb * (fingers > 0)
    h = xp.clip(box[2] - lay.z_rest, 0.0, 0.10)
    pushed = xp.linalg.norm(box[:2] - box_xy0) * (h < 0.01)
    success = held * (box[2] - lay.z_rest > LIFT_OK)
    r = (0.5 * (1 - xp.tanh(10 * d_palm)) + 0.5 * (1 - xp.tanh(10 * d_tips))
         + 0.5 * thumb + 0.25 * xp.minimum(fingers, 2.0)
         + 10.0 * held * h / 0.10 + 2.0 * success
         - 5.0 * pushed - 0.01 * xp.sum(action ** 2))
    return r, success


def draw_episode(rng: np.random.Generator) -> dict:
    """Box pose, perception bias and physics of one episode (numpy twin; the MJX env draws the same laws)."""
    return {"xy": rng.uniform(-BOX_XY_RANGE, BOX_XY_RANGE, 2), "yaw": rng.uniform(-BOX_YAW_RANGE, BOX_YAW_RANGE),
            "bias": np.r_[rng.normal(0, BIAS_XY, 2), rng.normal(0, BIAS_Z), rng.normal(0, BIAS_YAW)],
            "grip_friction": rng.uniform(*GRIP_FRICTION), "mass": rng.uniform(*BOX_MASS)}


def touch_np(m: mujoco.MjModel, d: mujoco.MjData, box_geoms, part_of) -> np.ndarray:
    """Which hand parts touch the box (thumb, index .. pinky, palm); `part_of`: geom -> part index or None."""
    out = np.zeros(6)
    for i in range(d.ncon):
        c = d.contact[i]
        if c.dist > 0.001:
            continue
        other = c.geom2 if c.geom1 in box_geoms else c.geom1 if c.geom2 in box_geoms else None
        if other is None:
            continue
        k = part_of(other)
        if k is not None and k >= 0:
            out[k] = 1.0
    return out


# ---------------------------------------------------------------------------------------- plain MuJoCo twin
class FlatGraspSim:
    """CPU twin of the training env: same model, same laws, same observation."""

    def __init__(self, xml: str | Path, frame_json: str | Path):
        self.model = mujoco.MjModel.from_xml_path(str(xml))
        self.data = mujoco.MjData(self.model)
        self.lay = Layout.of(self.model, frame_json)
        self.n_sub = round(DT / self.model.opt.timestep)
        self._mass0 = self.model.body_mass[self.lay.box_body]
        self._inertia0 = self.model.body_inertia[self.lay.box_body].copy()

    def reset(self, seed: int) -> np.ndarray:
        m, d, lay = self.model, self.data, self.lay
        self.rng = np.random.default_rng(seed)
        ep = self.ep = draw_episode(self.rng)
        m.geom_friction[lay.pad_geoms, 0] = ep["grip_friction"]
        m.body_mass[lay.box_body] = ep["mass"]
        m.body_inertia[lay.box_body] = self._inertia0 * ep["mass"] / self._mass0
        # masses changed: the derived constants too (the constraint solver's inverse weights; stale ones made
        # grasped boxes jump and the simulation diverge)
        mujoco.mj_setConst(m, d)
        mujoco.mj_resetData(m, d)
        d.qpos[:] = lay.qpos0
        a = lay.box_qadr
        d.qpos[a:a + 3] = [*(lay.box_xy + ep["xy"]), lay.z_rest + 0.0005]
        d.qpos[a + 3:a + 7] = [np.cos(ep["yaw"] / 2), 0, 0, np.sin(ep["yaw"] / 2)]
        self.ctrl = lay.ctrl0.copy()
        d.ctrl[:] = self.ctrl
        mujoco.mj_forward(m, d)
        self.box_xy0 = d.qpos[a:a + 2].copy()
        self.t = 0
        return self.observe()

    # the controller-facing interface (the robot twin in `flat_grasp_deploy` has the same)
    def arm_q(self) -> np.ndarray:
        return self.data.qpos[self.lay.act_qadr[:N_ARM]].copy()

    def touch(self) -> np.ndarray:
        return touch_np(self.model, self.data, {self.lay.box_geom}, lambda g: self.lay.part_of_geom[g])

    def box_seen(self) -> tuple[np.ndarray, float]:
        a, b = self.lay.box_qadr, self.ep["bias"]
        pos = self.data.qpos[a:a + 3] + b[:3] + self.rng.normal(0, NOISE, 3)
        return pos, float(box_yaw(np, self.data.qpos[a + 3:a + 7]) + b[3])

    def box_true(self) -> np.ndarray:
        return self.data.qpos[self.lay.box_qadr:self.lay.box_qadr + 3].copy()

    def observe(self) -> np.ndarray:
        d = self.data
        pos, yaw = self.box_seen()
        return observe(np, self.lay, d.qpos, d.qvel, d.site_xpos, d.site_xmat, self.ctrl, self.touch(), pos, yaw,
                       self.t / EPISODE_STEPS).astype(np.float32)

    def step(self, action: np.ndarray | None = None, ctrl: np.ndarray | None = None, on_substep=None):
        """One 50 Hz step: from a policy action (rates) or, for the expert, from position targets directly."""
        lay = self.lay
        if ctrl is None:
            action = np.clip(action, -1, 1)
            ctrl = self.ctrl + action * RATE * DT
        else:
            action = np.zeros(lay.lo.size)
        self.ctrl = np.clip(ctrl, lay.lo, lay.hi)
        self.data.ctrl[:] = self.ctrl
        for _ in range(self.n_sub):
            mujoco.mj_step(self.model, self.data)
            if on_substep is not None:
                on_substep()
        self.t += 1
        r, success = reward(np, lay, self.data.qpos, self.data.site_xpos, self.touch(), action, self.box_xy0)
        return self.observe(), float(r), bool(success), self.t >= EPISODE_STEPS

    def lift_cm(self) -> float:
        return 100 * (self.box_true()[2] - self.lay.z_rest)

    def success(self) -> bool:
        t = self.touch()
        return bool(t[0] and t[1:5].any() and self.lift_cm() / 100 > LIFT_OK)


class Expert:
    """The expert's top grasp, planned once from the observed pose (the first frame), as `skills.top_grasp_plan`:
    the palm turns to the box's heading and moves over it (the pre-grasp swung about the box), comes down 10 cm,
    the hand closes, the palm lifts 15 cm. Arm joint targets from the arm IK at the key poses, interpolated in
    joint space (the descent and lift through intermediate poses, so the palm moves straight)."""

    def __init__(self, twin, ik=None):
        from homehand.control.ik import ArmIK
        lay = twin.lay
        self.ik = ik or ArmIK()
        pos, yaw = twin.box_seen()
        yaw = float(np.clip((yaw + np.pi / 2) % np.pi - np.pi / 2, -0.8, 0.8))
        b0 = np.r_[lay.box_xy, lay.z_rest]
        c, s = np.cos(yaw), np.sin(yaw)
        Rz = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
        p_above = pos + Rz @ (lay.palm_pos - b0)
        p_above[2] = lay.palm_pos[2] + (pos[2] - b0[2])
        Rp = np.zeros(9)
        mujoco.mju_quat2Mat(Rp, lay.palm_quat)
        quat = np.zeros(4)
        mujoco.mju_mat2Quat(quat, (Rz @ Rp.reshape(3, 3)).reshape(-1))
        q = self.ik.q_work.copy()
        adr = self.ik.arm_qadr["right"]
        q[adr] = twin.arm_q()
        keys = [(0.0, q[adr].copy())]
        down = [p_above + np.array([0, 0, -0.10 * f]) for f in (0.25, 0.5, 0.75, 1.0)]
        up = [p_above + np.array([0, 0, -0.10 + 0.15 * f]) for f in (0.25, 0.5, 0.75, 1.0)]
        for t, p in [(1.0, p_above)] + [(1.0 + 0.25 * (k + 1), p) for k, p in enumerate(down)] \
                + [(3.0 + 0.25 * (k + 1), p) for k, p in enumerate(up)]:
            q, _ = self.ik.solve(q, {"right": (p, quat)}, iters=100, converge_orientation=True)
            keys.append((t, q[adr].copy()))
        keys.insert(6, (3.0, keys[5][1]))           # hold the grasp pose while the hand closes (2-3 s)
        self.times = np.array([k[0] for k in keys])
        self.arm = np.array([k[1] for k in keys])

    def ctrl(self, t: float) -> np.ndarray:
        arm = np.array([np.interp(t, self.times, self.arm[:, j]) for j in range(N_ARM)])
        f = float(np.clip(t - 2.0, 0, 1))
        return np.r_[arm, FINGER_OPEN + (FINGER_CLOSE - FINGER_OPEN) * f]


def run_expert(twin, seed: int, ik=None, **kw) -> bool:
    twin.reset(seed)
    ex = Expert(twin, ik)
    for k in range(EPISODE_STEPS):
        twin.step(ctrl=ex.ctrl(k * DT), **kw)
    return twin.success()


# ---------------------------------------------------------------------------------------- MJX (training)
def make_mjx_env(xml: str | Path, frame_json: str | Path):
    """Brax environment on MJX (imports jax / brax here: the plain-MuJoCo twin above needs neither)."""
    import jax
    import jax.numpy as jp
    from brax.envs.base import Env, State
    from mujoco import mjx

    class FlatGraspEnv(Env):
        def __init__(self):
            self.mj_model = mujoco.MjModel.from_xml_path(str(xml))
            self.lay = Layout.of(self.mj_model, frame_json)
            self.sys = mjx.put_model(self.mj_model)
            self.n_sub = round(DT / self.mj_model.opt.timestep)
            self._rate = jp.asarray(RATE * DT)

        @property
        def observation_size(self):
            return OBS_SIZE

        @property
        def action_size(self):
            return int(self.lay.lo.size)

        @property
        def backend(self):
            return "mjx"

        def _touch(self, data):
            c = data._impl.contact
            lay = self.lay
            g0, g1 = c.geom[:, 0], c.geom[:, 1]
            on_box = ((g0 == lay.box_geom) | (g1 == lay.box_geom)) & (c.dist <= 0.001)
            other = jp.where(g0 == lay.box_geom, g1, g0)
            part = jp.asarray(lay.part_of_geom)[other]
            return (on_box[:, None] & (part[:, None] == jp.arange(6)[None, :])).any(axis=0).astype(jp.float32)

        def _obs(self, data, info, rng, t):
            a = self.lay.box_qadr
            b = info["bias"]
            pos = data.qpos[a:a + 3] + b[:3] + NOISE * jax.random.normal(rng, (3,))
            yaw = box_yaw(jp, data.qpos[a + 3:a + 7]) + b[3]
            return observe(jp, self.lay, data.qpos, data.qvel, data.site_xpos, data.site_xmat, info["ctrl"],
                           self._touch(data), pos, yaw, t / EPISODE_STEPS)

        def reset(self, rng):
            lay = self.lay
            rng, k1, k2, k3, k4 = jax.random.split(rng, 5)
            xy = jp.asarray(lay.box_xy) + jax.random.uniform(k1, (2,), minval=-BOX_XY_RANGE, maxval=BOX_XY_RANGE)
            yaw = jax.random.uniform(k2, (), minval=-BOX_YAW_RANGE, maxval=BOX_YAW_RANGE)
            bias = jax.random.normal(k3, (4,)) * jp.array([BIAS_XY, BIAS_XY, BIAS_Z, BIAS_YAW])
            a = lay.box_qadr
            qpos = jp.asarray(lay.qpos0)
            qpos = qpos.at[a:a + 3].set(jp.array([xy[0], xy[1], lay.z_rest + 0.0005]))
            qpos = qpos.at[a + 3:a + 7].set(jp.array([jp.cos(yaw / 2), 0.0, 0.0, jp.sin(yaw / 2)]))
            ctrl = jp.asarray(lay.ctrl0, dtype=jp.float32)
            data = mjx.make_data(self.sys).replace(qpos=qpos, ctrl=ctrl)
            data = mjx.forward(self.sys, data)
            info = {"rng": rng, "ctrl": ctrl, "bias": bias, "box_xy0": xy, "t": jp.zeros(())}
            obs = self._obs(data, info, k4, 0.0)
            metrics = {"success": jp.zeros(()), "lift_cm": jp.zeros(())}
            return State(data, obs, jp.zeros(()), jp.zeros(()), metrics, info)

        def step(self, state, action):
            lay = self.lay
            action = jp.clip(action, -1, 1)
            ctrl = jp.clip(state.info["ctrl"] + action * self._rate, lay.lo, lay.hi)
            data = state.pipeline_state.replace(ctrl=ctrl)
            data = jax.lax.fori_loop(0, self.n_sub, lambda _, d: mjx.step(self.sys, d), data)
            rng, k = jax.random.split(state.info["rng"])
            t = state.info["t"] + 1
            info = {**state.info, "rng": rng, "ctrl": ctrl, "t": t}
            r, success = reward(jp, lay, data.qpos, data.site_xpos, self._touch(data), action,
                                state.info["box_xy0"])
            obs = self._obs(data, info, k, t)
            box_z = data.qpos[lay.box_qadr + 2]
            bad = jp.isnan(data.qpos).any() | jp.isnan(obs).any() | (box_z < TABLE_Z - 0.05)
            obs = jp.where(jp.isnan(obs), 0.0, obs)
            r = jp.where(jp.isnan(r), 0.0, r)
            metrics = {**state.metrics, "success": success.astype(jp.float32),
                       "lift_cm": 100 * (box_z - lay.z_rest)}
            return state.replace(pipeline_state=data, obs=obs, reward=r, done=bad.astype(jp.float32),
                                 metrics=metrics, info=info)

    return FlatGraspEnv()


def randomize(sys, rng, lay: Layout):
    """Per-env physics for brax's domain-randomisation wrapper: grip friction and box mass (inertia scaled)."""
    import jax

    mass0 = sys.body_mass[lay.box_body]
    pads = np.asarray(lay.pad_geoms)

    box_dofs = np.arange(6) + int(lay.box_dadr)

    @jax.vmap
    def one(r):
        k1, k2 = jax.random.split(r)
        f = jax.random.uniform(k1, (), minval=GRIP_FRICTION[0], maxval=GRIP_FRICTION[1])
        mass = jax.random.uniform(k2, (), minval=BOX_MASS[0], maxval=BOX_MASS[1])
        # the derived inverse weights scale with 1 / mass too (mj_setConst on the CPU)
        w = mass0 / mass
        return (sys.geom_friction.at[pads, 0].set(f), sys.body_mass.at[lay.box_body].set(mass),
                sys.body_inertia.at[lay.box_body].set(sys.body_inertia[lay.box_body] / w),
                sys.body_invweight0.at[lay.box_body].set(sys.body_invweight0[lay.box_body] * w),
                sys.dof_invweight0.at[box_dofs].set(sys.dof_invweight0[box_dofs] * w))

    fields = ("geom_friction", "body_mass", "body_inertia", "body_invweight0", "dof_invweight0")
    values = one(rng)
    in_axes = jax.tree_util.tree_map(lambda x: None, sys)
    in_axes = in_axes.tree_replace({k: 0 for k in fields})
    sys = sys.tree_replace(dict(zip(fields, values)))
    return sys, in_axes

In [ ]:
%%writefile train_flat_grasp.py
"""PPO (Brax) on the MJX flat-box grasping env. Runs where jax + brax + mujoco-mjx are (a GPU: Kaggle; see
kaggle/flat_grasp_ppo.ipynb). Needs only this folder's `flat_grasp_env.py` and the two generated files.

    python train_flat_grasp.py --xml flat_grasp_train.xml --frame flat_grasp.json --steps 60000000 --out out/

Writes out/params.pkl (the latest policy, also every evaluation), out/progress.jsonl (one line per evaluation) and
out/policy.npz (the policy for numpy inference: `flat_grasp_policy.Policy`).
"""

from __future__ import annotations

import argparse
import functools
import json
import pickle
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import flat_grasp_env as E  # noqa: E402

PPO = dict(num_envs=2048, batch_size=512, num_minibatches=32, unroll_length=10, num_updates_per_batch=8,
           discounting=0.97, learning_rate=3e-4, entropy_cost=1e-2, reward_scaling=1.0, clipping_epsilon=0.2,
           normalize_observations=True, max_grad_norm=1.0)
POLICY_LAYERS = (256, 256, 128)
VALUE_LAYERS = (256, 256, 256)


def export_npz(params, path: Path) -> None:
    """Observation normaliser + policy MLP as plain arrays (brax: swish hidden layers, tanh(mean) for the
    deterministic action)."""
    norm, policy = params[0], params[1]
    arrays = {"obs_mean": np.asarray(norm.mean), "obs_std": np.asarray(norm.std)}
    layers = policy["params"]
    for k in range(len(layers)):
        arrays[f"w{k}"] = np.asarray(layers[f"hidden_{k}"]["kernel"])
        arrays[f"b{k}"] = np.asarray(layers[f"hidden_{k}"]["bias"])
    np.savez(path, **arrays)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--xml", default="flat_grasp_train.xml")
    ap.add_argument("--frame", default="flat_grasp.json")
    ap.add_argument("--steps", type=int, default=60_000_000)
    ap.add_argument("--evals", type=int, default=20)
    ap.add_argument("--num-envs", type=int, default=PPO["num_envs"])
    ap.add_argument("--batch-size", type=int, default=PPO["batch_size"])
    ap.add_argument("--minibatches", type=int, default=PPO["num_minibatches"])
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="out")
    a = ap.parse_args()

    import jax
    from brax.training.agents.ppo import networks as ppo_networks
    from brax.training.agents.ppo import train as ppo

    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    env = E.make_mjx_env(a.xml, a.frame)
    eval_env = E.make_mjx_env(a.xml, a.frame)
    print("devices:", jax.devices(), flush=True)
    t0 = time.time()

    def progress(step, metrics):
        row = {"step": int(step), "time_s": round(time.time() - t0, 1)}
        row.update({k: float(v) for k, v in metrics.items() if k.startswith("eval/") and np.ndim(v) == 0})
        with open(out / "progress.jsonl", "a") as f:
            f.write(json.dumps(row) + "\n")
        print(f"{step:>11,d}  {row['time_s']:>7.0f}s  reward {row.get('eval/episode_reward', 0):8.1f}  "
              f"success-steps {row.get('eval/episode_success', 0):6.1f}  "
              f"lift {row.get('eval/episode_lift_cm', 0) / E.EPISODE_STEPS:5.2f} cm", flush=True)

    def save(step, make_policy, params):
        with open(out / "params.pkl", "wb") as f:
            pickle.dump(jax.tree_util.tree_map(np.asarray, params), f)

    cfg = dict(PPO, num_envs=a.num_envs, batch_size=a.batch_size, num_minibatches=a.minibatches)
    make_policy, params, _ = ppo.train(
        environment=env, eval_env=eval_env, num_timesteps=a.steps, num_evals=a.evals,
        episode_length=E.EPISODE_STEPS, action_repeat=1, seed=a.seed,
        network_factory=functools.partial(ppo_networks.make_ppo_networks,
                                          policy_hidden_layer_sizes=POLICY_LAYERS,
                                          value_hidden_layer_sizes=VALUE_LAYERS),
        randomization_fn=functools.partial(E.randomize, lay=env.lay),
        progress_fn=progress, policy_params_fn=save, **cfg)
    save(None, make_policy, params)
    export_npz(params, out / "policy.npz")
    (out / "config.json").write_text(json.dumps({"ppo": cfg, "steps": a.steps, "policy_layers": POLICY_LAYERS,
                                                 "value_layers": VALUE_LAYERS, "seed": a.seed}, indent=2))
    print(f"done in {time.time() - t0:.0f} s", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile mjx_bench.py
"""Throughput of the flat-grasp env on MJX: policy steps per second for a batch of environments (random actions).

    python mjx_bench.py flat_grasp_train.xml [more.xml ...] --envs 2048
"""

from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import flat_grasp_env as E  # noqa: E402


def main() -> None:
    import jax
    import jax.numpy as jp
    ap = argparse.ArgumentParser()
    ap.add_argument("xml", nargs="+")
    ap.add_argument("--frame", default="flat_grasp.json")
    ap.add_argument("--envs", type=int, default=2048)
    ap.add_argument("--steps", type=int, default=20)
    a = ap.parse_args()
    for xml in a.xml:
        env = E.make_mjx_env(xml, a.frame)
        reset, step = jax.jit(jax.vmap(env.reset)), jax.jit(jax.vmap(env.step))
        t0 = time.time()
        s = reset(jax.random.split(jax.random.PRNGKey(0), a.envs))
        act = jax.random.uniform(jax.random.PRNGKey(1), (a.envs, env.action_size), minval=-1, maxval=1)
        s = step(s, act)
        jax.block_until_ready(s.obs)
        compile_s = time.time() - t0
        t0 = time.time()
        for _ in range(a.steps):
            s = step(s, act)
        jax.block_until_ready(s.obs)
        dt = time.time() - t0
        print(f"{xml}: compile {compile_s:.0f} s, {a.envs * a.steps / dt:,.0f} policy steps/s "
              f"(1 device, {a.envs} envs)", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile flat_grasp_train.xml
<mujoco model="flat_grasp">
  <compiler angle="radian" meshdir="assets/" texturedir="assets/"/>

  <option timestep="0.004" impratio="10" integrator="implicitfast" cone="elliptic" iterations="10" ls_iterations="20"/>

  <asset>
    <mesh name="geom31_hull" vertex="-0.0242833 -0.00692377 -0.0694194 -0.0241152 0.00551586 -0.0691178 -0.0218137 0.0113447 -0.0689978 -0.0191091 -0.0382481 0.0491708 -0.0190719 -0.0357583 0.0549932 -0.00850511 -0.0252177 -0.0700167 -0.00784827 0.0233844 -0.0688383 -0.00757239 0.0382385 0.0537007 -0.00743612 0.0398745 0.011904 -0.00503561 -0.0341206 0.070057 -0.0012846 -0.0346699 0.0707508 0.00381403 0.0315282 0.0696259 0.00579139 -0.0422307 0.0583752 0.00947544 -0.0362272 0.0642926 0.00993181 -0.0254624 -0.0701995 0.0135234 -0.0256575 -0.062136 0.0160985 0.0201543 -0.0691468 0.017873 0.034701 0.0588653 0.0185057 0.0274739 0.0638053 0.0213925 0.0236969 0.0575775 0.0238972 -0.0134227 -0.07004 0.0242237 0.0107336 -0.0694543 0.0270382 -0.0013841 -0.0697767"/>
    <mesh name="geom33_hull" vertex="-0.0149762 -0.00321083 -0.00556432 -0.0141218 -0.00496041 -0.00485969 -0.0133395 0.00827268 -0.00541568 -0.0120417 0.010358 -0.0047281 -0.00975842 -0.00385196 -0.0134752 -0.00821495 -0.0062034 -0.0122673 -0.00785366 0.00845343 -0.0132171 -0.00682466 0.00971674 -0.0126406 -0.00614204 0.0100593 -0.0122178 -0.0054153 0.010169 -0.0117472 -0.00150696 -0.00529082 0.00708891 -0.00143798 0.00617386 0.00620526 -0.00073254 0.00619583 0.00654531 0.000748679 -0.00527119 0.0075196 0.00220496 0.00613164 0.00594185 0.00264771 -0.00538165 0.00623476 0.00271668 0.00608303 0.00535111 0.00360897 -0.00614232 -0.00355931 0.00422519 -0.00605112 -0.00232798 0.00429417 0.00541356 -0.00321163 0.0047066 -0.00574118 0.00173086"/>
    <mesh name="geom35_hull" vertex="-0.00894703 -0.00122466 -0.0325257 -0.00875397 0.00372278 -0.0325321 -0.00840841 0.00659145 -0.0316416 -0.00668139 -0.0136872 0.0157715 -0.00602288 0.0133278 -0.0265249 -0.00546383 0.0124505 0.0272511 -0.0052552 -0.00865342 0.0299402 -0.00525436 -0.00743044 0.0307322 -0.00489152 0.0101249 0.0342963 -0.00464574 -0.0155096 -0.0205865 -0.00299223 0.0111982 0.0344233 0.00153968 -0.0140968 0.0225897 0.00350156 0.0147849 -0.0239386 0.00454288 0.0105691 0.0342787 0.00490711 0.0101028 0.0342985 0.00524402 -0.00866023 0.0299395 0.00524486 -0.00743724 0.0307316 0.00584038 -0.015424 -0.0184238 0.00758945 0.00983546 -0.029819 0.00768817 0.00980379 -0.0297322 0.0086871 0.00173102 -0.0328517 0.00874317 -0.00266007 -0.0318009 0.0088761 -0.00262652 -0.0317487 0.00949223 -0.00615998 0.0208962"/>
    <mesh name="geom37_hull" vertex="-0.00784741 0.00833412 -0.0116747 -0.00762747 -0.0100173 -0.00545102 -0.00760053 -0.00828414 -0.0140219 -0.00754308 -0.00397311 -0.020371 -0.00735309 -0.00235035 0.0216858 -0.00734498 -0.000867563 0.0215284 -0.00723186 -0.000662379 -0.0201278 -0.00717405 -0.00350854 0.021279 -0.00708141 0.00913201 -0.0123218 -0.00505266 0.00839847 0.0134083 -0.0031925 0.00834589 0.0169847 3.8289e-06 -0.00854524 0.0135155 0.00471066 0.0113049 -0.00504092 0.00479164 0.0074778 0.0168872 0.00706769 0.00572185 -0.0165992 0.00717222 -0.00351028 0.0212786 0.00727894 -0.00369107 -0.0207005 0.00731345 -0.00277072 0.021619 0.00736109 -0.0022148 0.0216908 0.0073691 -0.00155507 0.0216554 0.00751537 0.00884731 -0.012181 0.00755166 -0.00399731 -0.0203461 0.00771623 -0.00993502 -0.00588221"/>
    <mesh name="geom39_hull" vertex="-0.00552798 -0.00354921 0.000292864 -0.0053296 0.00354961 -0.000968028 -0.00511035 0.00354855 0.00280331 -0.00489902 -0.00339599 0.00327132 -0.00416329 -0.00354825 -0.00323159 0.000267262 0.00304724 0.00851103 0.000333295 -0.00305197 0.00853666 0.000463172 0.003547 0.00805156 0.00401667 -0.00648859 -0.00857428 0.00417451 -0.00632859 0.00481762 0.00565209 -0.00413767 0.00791454 0.00580988 -0.00368958 0.00806685 0.00592656 0.00310125 0.00818312 0.00623777 0.00603244 -0.011296 0.00982913 0.00341953 -0.0157532 0.0116817 0.00477572 -0.00151255 0.0131032 0.00604815 -0.0113613 0.0141293 -0.00433642 -0.0164265 0.0152953 0.00426411 -0.0161765 0.0157317 0.00163305 -0.0170792 0.017961 -0.00428126 -0.0135656 0.0191982 -0.00220264 -0.0146375 0.0198534 0.00210598 -0.0135683 0.0201227 -0.00156694 -0.0125891"/>
    <mesh name="geom41_hull" vertex="-0.0106262 0.00211235 0.0214416 -0.0104991 0.00104433 0.0216508 -0.0104702 0.00732941 0.0180821 -0.00998924 0.00840316 0.0164662 -0.00943221 0.00554713 -0.0215818 -0.00929079 0.00600106 -0.0211816 -0.00917949 0.00129586 -0.022631 -0.00914385 0.00383843 -0.0230025 -0.00825727 -0.00531164 0.0215763 -0.00774286 -0.00871074 0.0168325 -0.00418143 -0.00785954 -0.0219816 -0.00273673 -0.0108916 0.01282 -0.00120583 -0.00767594 -0.022003 -0.000994393 0.0104106 0.005222 0.000846792 -0.00750172 -0.0198149 0.00148657 -0.00484291 0.021736 0.00192303 -0.0058927 -0.0205843 0.00216136 0.00396902 -0.0231 0.0022953 0.00559304 -0.022416 0.00231669 -0.00442125 0.0215433 0.00241868 0.0062762 -0.0214913 0.00264853 0.00808553 0.0182462 0.00325815 -8.94478e-05 0.0218144 0.00336983 0.00299435 0.0215472"/>
    <mesh name="geom43_hull" vertex="-0.011643 -0.00300141 0.0308093 -0.0113328 0.00300398 0.0327399 -0.0106139 -0.00295387 0.0345591 -0.00690743 0.00295578 0.0381816 -0.00690134 -0.00300198 0.0381683 -0.00639951 0.00303769 0.0383325 -0.00536472 -0.00295449 0.0385123 -0.00501821 0.00218075 -0.00499429 -0.00386341 0.00295543 0.0380262 -0.00310161 -0.00300242 0.0372948 -0.00224955 -0.00614671 0.0307359 -0.00189277 -0.00279749 -0.0123261 -0.000907673 0.00411894 -0.0121385 -0.000766985 -0.0042235 -0.0120375 -0.000427538 0.00283306 -0.0135104 0.000309906 0.00535278 -0.00373239 0.00102506 0.00483454 0.029835 0.00197684 -0.0028233 -0.0136161 0.00262941 0.000931536 -0.0140536 0.00286914 0.00282275 -0.0132938 0.00305305 -0.0038479 -0.0109511 0.00385668 -0.00201299 -0.0129796 0.00401119 0.00251274 -0.0121856 0.00406357 -0.00101149 0.0212313"/>
    <mesh name="geom45_hull" vertex="-0.0106262 0.00211235 0.0214416 -0.0104991 0.00104433 0.0216508 -0.0104702 0.00732941 0.0180821 -0.00998924 0.00840316 0.0164662 -0.00943221 0.00554713 -0.0215818 -0.00929079 0.00600106 -0.0211816 -0.00917949 0.00129586 -0.022631 -0.00914385 0.00383843 -0.0230025 -0.00825727 -0.00531164 0.0215763 -0.00774286 -0.00871074 0.0168325 -0.00418143 -0.00785954 -0.0219816 -0.00273673 -0.0108916 0.01282 -0.00120583 -0.00767594 -0.022003 -0.000994393 0.0104106 0.005222 0.000846792 -0.00750172 -0.0198149 0.00148657 -0.00484291 0.021736 0.00192303 -0.0058927 -0.0205843 0.00216136 0.00396902 -0.0231 0.0022953 0.00559304 -0.022416 0.00231669 -0.00442125 0.0215433 0.00241868 0.0062762 -0.0214913 0.00264853 0.00808553 0.0182462 0.00325815 -8.94478e-05 0.0218144 0.00336983 0.00299435 0.0215472"/>
    <mesh name="geom47_hull" vertex="-0.00487771 0.00216496 -0.0245152 -0.00479726 -0.00547506 0.017208 -0.00322113 -0.00141501 -0.0289738 -0.00305116 -0.00129343 0.0261337 -0.00305003 -5.40316e-05 0.0265455 -0.00304885 0.00124285 0.0264048 -0.00304446 0.00607974 0.0220025 -0.00304411 0.00646474 0.0213895 -0.00279345 0.00310921 -0.0274098 -0.002387 0.00050958 -0.0297058 -0.00215142 -0.00311161 -0.0281518 0.000184169 -0.00739451 0.0160847 0.00139818 0.00745237 0.0155386 0.00146336 -0.0021821 -0.0297309 0.001746 -9.19799e-05 -0.0300534 0.00304759 -0.00129897 0.0261337 0.00304872 -5.95663e-05 0.0265455 0.00304989 0.00123732 0.0264048 0.00305464 0.0064592 0.0213895 0.00312817 0.00195093 -0.0286836 0.00321869 -0.00142085 -0.0289738 0.00369109 -0.00593202 -0.00811768 0.00448912 0.00233243 -0.0260279 0.00617699 -0.0019945 0.0184421"/>
    <mesh name="geom49_hull" vertex="-0.0106262 0.00211235 0.0214416 -0.0104991 0.00104433 0.0216508 -0.0104702 0.00732941 0.0180821 -0.00998924 0.00840316 0.0164662 -0.00943221 0.00554713 -0.0215818 -0.00929079 0.00600106 -0.0211816 -0.00917949 0.00129586 -0.022631 -0.00914385 0.00383843 -0.0230025 -0.00825727 -0.00531164 0.0215763 -0.00774286 -0.00871074 0.0168325 -0.00418143 -0.00785954 -0.0219816 -0.00273673 -0.0108916 0.01282 -0.00120583 -0.00767594 -0.022003 -0.000994393 0.0104106 0.005222 0.000846792 -0.00750172 -0.0198149 0.00148657 -0.00484291 0.021736 0.00192303 -0.0058927 -0.0205843 0.00216136 0.00396902 -0.0231 0.0022953 0.00559304 -0.022416 0.00231669 -0.00442125 0.0215433 0.00241868 0.0062762 -0.0214913 0.00264853 0.00808553 0.0182462 0.00325815 -8.94478e-05 0.0218144 0.00336983 0.00299435 0.0215472"/>
    <mesh name="geom51_hull" vertex="-0.011643 -0.00300141 0.0308093 -0.0113328 0.00300398 0.0327399 -0.0106139 -0.00295387 0.0345591 -0.00690743 0.00295578 0.0381816 -0.00690134 -0.00300198 0.0381683 -0.00639951 0.00303769 0.0383325 -0.00536472 -0.00295449 0.0385123 -0.00501821 0.00218075 -0.00499429 -0.00386341 0.00295543 0.0380262 -0.00310161 -0.00300242 0.0372948 -0.00224955 -0.00614671 0.0307359 -0.00189277 -0.00279749 -0.0123261 -0.000907673 0.00411894 -0.0121385 -0.000766985 -0.0042235 -0.0120375 -0.000427538 0.00283306 -0.0135104 0.000309906 0.00535278 -0.00373239 0.00102506 0.00483454 0.029835 0.00197684 -0.0028233 -0.0136161 0.00262941 0.000931536 -0.0140536 0.00286914 0.00282275 -0.0132938 0.00305305 -0.0038479 -0.0109511 0.00385668 -0.00201299 -0.0129796 0.00401119 0.00251274 -0.0121856 0.00406357 -0.00101149 0.0212313"/>
    <mesh name="geom53_hull" vertex="-0.0106262 0.00211235 0.0214416 -0.0104991 0.00104433 0.0216508 -0.0104702 0.00732941 0.0180821 -0.00998924 0.00840316 0.0164662 -0.00943221 0.00554713 -0.0215818 -0.00929079 0.00600106 -0.0211816 -0.00917949 0.00129586 -0.022631 -0.00914385 0.00383843 -0.0230025 -0.00825727 -0.00531164 0.0215763 -0.00774286 -0.00871074 0.0168325 -0.00418143 -0.00785954 -0.0219816 -0.00273673 -0.0108916 0.01282 -0.00120583 -0.00767594 -0.022003 -0.000994393 0.0104106 0.005222 0.000846792 -0.00750172 -0.0198149 0.00148657 -0.00484291 0.021736 0.00192303 -0.0058927 -0.0205843 0.00216136 0.00396902 -0.0231 0.0022953 0.00559304 -0.022416 0.00231669 -0.00442125 0.0215433 0.00241868 0.0062762 -0.0214913 0.00264853 0.00808553 0.0182462 0.00325815 -8.94478e-05 0.0218144 0.00336983 0.00299435 0.0215472"/>
    <mesh name="geom55_hull" vertex="-0.00507744 0.0022008 -0.0188958 -0.00484256 -0.00539609 0.0122181 -0.0036672 -0.000400089 -0.0225873 -0.00302352 -0.000382285 0.0214463 -0.00301002 0.00145836 -0.0231774 -0.00299294 0.00666502 0.0151056 -0.00295275 -0.00174572 0.0208582 -0.00292191 0.00110537 0.0214292 -0.00285519 0.0049641 0.018756 -0.00283278 0.00325002 -0.0217569 -0.00214286 -0.0030563 -0.0217538 5.40189e-06 -0.00732677 0.01091 0.00151631 0.00702688 0.0101568 0.00174504 -0.00223566 -0.0229836 0.00238729 0.000436042 -0.0235249 0.00285492 -0.000401711 0.0215463 0.00295283 0.000610643 0.0215342 0.00295297 -0.00174537 0.0208582 0.0029922 0.00568806 0.0176119 0.00314368 0.00235493 -0.022652 0.00360153 -0.0013126 -0.022274 0.00363159 -0.00591466 -0.00601433 0.00472366 0.00247969 -0.0200513 0.00622951 -0.00189506 0.013496"/>
  </asset>

  <worldbody>
    <geom name="floor" size="3 3 0.05" type="plane" contype="0" conaffinity="0" rgba="0.7 0.7 0.7 1"/>
    <geom name="counter" size="0.35 0.65 0.4" pos="0.55 0 0.4" type="box" contype="0" conaffinity="0" rgba="0.93 0.92 0.89 1"/>
    <geom name="table" size="0.35 0.65 0.01" pos="0.55 0 0.8" type="plane" contype="4" group="3" friction="0.8 0.01 0.001" rgba="0 0 0 0"/>
    <light pos="0.5 0 2.5" type="directional"/>
    <body name="pelvis" pos="0 0 0.793">
      <inertial pos="0 0 -0.07605" quat="1 0 -0.000399148 0" mass="3.813" diaginertia="0.010549 0.0093089 0.0079184"/>
      <body name="waist_yaw_link">
        <inertial pos="0.003964 0 0.018769" quat="-0.0178291 0.628464 0.0282471 0.777121" mass="0.244" diaginertia="0.000158561 0.000124229 9.67669e-05"/>
        <body name="waist_roll_link" pos="-0.0039635 0 0.035">
          <inertial pos="0 -0.000236 0.010111" quat="0.99979 0.020492 0 0" mass="0.047" diaginertia="7.515e-06 6.40206e-06 3.98394e-06"/>
          <body name="torso_link" pos="0 0 0.019">
            <inertial pos="0.00331658 0.000261533 0.179856" quat="0.999831 0.000376204 0.0179895 -0.00377704" mass="9.598" diaginertia="0.12407 0.111951 0.0325382"/>
            <site name="imu" size="0.01" pos="-0.03959 -0.00224 0.13792"/>
            <camera name="head_camera" quat="0.651548 0.275251 -0.27547 -0.651029" pos="0.06 0 0.45" fovy="90"/>
            <body name="right_shoulder_pitch_link" pos="0.0039563 -0.10021 0.23778" quat="0.990264 -0.139201 1.38722e-05 9.86868e-05" gravcomp="1">
              <inertial pos="0 -0.035892 -0.011628" quat="0.68225 -0.326267 0.0130458 0.654152" mass="0.718" diaginertia="0.000465864 0.000432842 0.000406394"/>
              <joint name="right_shoulder_pitch_joint" range="-3.0892 2.6704" actuatorfrcrange="-25 25" armature="0.01" damping="1" frictionloss="0.3" axis="0 1 0"/>
              <geom name="geom13" size="0.03 0.025" pos="0 -0.04 -0.01" quat="0.707107 0 0.707107 0" type="cylinder" contype="0" conaffinity="0" group="3" rgba="0.7 0.7 0.7 1"/>
              <body name="right_shoulder_roll_link" pos="0 -0.038 -0.013831" quat="0.990268 0.139172 0 0" gravcomp="1">
                <inertial pos="-0.000227 -0.00727 -0.063243" quat="0.712604 -0.00710317 -0.0196223 0.701256" mass="0.643" diaginertia="0.000691311 0.000618011 0.000388977"/>
                <joint name="right_shoulder_roll_joint" range="-2.2515 1.5882" actuatorfrcrange="-25 25" armature="0.01" damping="1" frictionloss="0.3" axis="1 0 0"/>
                <geom name="geom15" size="0.03 0.015" pos="-0.004 -0.006 -0.053" type="cylinder" contype="0" conaffinity="0" group="3" rgba="0.7 0.7 0.7 1"/>
                <body name="right_shoulder_yaw_link" pos="0 -0.00624 -0.1032" gravcomp="1">
                  <inertial pos="0.010773 0.002949 -0.072009" quat="0.687134 -0.0679942 -0.0964829 0.716879" mass="0.734" diaginertia="0.00106187 0.00103217 0.000400661"/>
                  <joint name="right_shoulder_yaw_joint" range="-2.618 2.618" actuatorfrcrange="-25 25" armature="0.01" damping="1" frictionloss="0.3"/>
                  <body name="right_elbow_link" pos="0.015783 0 -0.080518" gravcomp="1">
                    <inertial pos="0.064956 -0.004454 -0.010062" quat="0.388129 0.388821 0.636132 0.541765" mass="0.6" diaginertia="0.000443035 0.000421612 0.000259353"/>
                    <joint name="right_elbow_joint" range="-1.0472 2.0944" actuatorfrcrange="-25 25" armature="0.01" damping="1" frictionloss="0.3" axis="0 1 0"/>
                    <body name="right_wrist_roll_link" pos="0.1 -0.00188791 -0.01" gravcomp="1">
                      <inertial pos="0.0171394 -0.000537591 4.8864e-07" quat="0.411667 0.575338 -0.411094 0.574906" mass="0.085445" diaginertia="5.48211e-05 4.96646e-05 3.57798e-05"/>
                      <joint name="right_wrist_roll_joint" range="-1.97222 1.97222" actuatorfrcrange="-25 25" armature="0.01" damping="0.3" frictionloss="0.3" axis="1 0 0"/>
                      <body name="right_wrist_pitch_link" pos="0.038 0 0" gravcomp="1">
                        <inertial pos="0.0229999 0.00111685 -0.00111658" quat="0.643608 0.293036 0.661363 0.249998" mass="0.48405" diaginertia="0.000430353 0.000429873 0.000164648"/>
                        <joint name="right_wrist_pitch_joint" range="-1.61443 1.61443" actuatorfrcrange="-5 5" armature="0.01" damping="0.3" frictionloss="0.3" axis="0 1 0"/>
                        <body name="right_wrist_yaw_link" pos="0.046 0 0" gravcomp="1">
                          <inertial pos="0.0885506 -0.00212216 0.000573742" quat="0.507224 0.511377 0.494292 0.486717" mass="0.457415" diaginertia="0.0010598 0.000895373 0.0003238"/>
                          <joint name="right_wrist_yaw_joint" range="-1.61443 1.61443" actuatorfrcrange="-5 5" armature="0.01" damping="0.3" frictionloss="0.3"/>
                          <body name="R_hand_base_link" pos="0.038 0 0" quat="9.38184e-07 0.707105 -0.707108 9.38187e-07" gravcomp="1">
                            <inertial pos="-0.0025264 -0.066047 0.0019598" quat="0.993005 -0.113528 -0.0210604 0.0246792" mass="0.14143" diaginertia="0.000122991 8.41186e-05 7.61951e-05"/>
                            <geom name="geom27" size="0.028 0.028 0.0139" pos="0 -0.0136 0" quat="0.707105 0.707108 0 0" type="box" contype="2" conaffinity="5" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1"/>
                            <geom name="geom28" size="0.01955 0.0101 0.0287" pos="-0.0032 -0.038 0" type="box" contype="2" conaffinity="5" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1"/>
                            <geom name="geom29" size="0.01955 0.02 0.0407" pos="-0.0032 -0.0682 0" type="box" contype="2" conaffinity="5" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1"/>
                            <geom name="geom30" size="0.0161 0.02 0.0407" pos="0 -0.1081 0" type="box" contype="2" conaffinity="5" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1"/>
                            <geom name="geom31" pos="0.00390141 -0.0692979 0.00244325" quat="0.713824 0.697085 -0.0513767 0.0434547" type="mesh" contype="2" conaffinity="5" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom31_hull"/>
                            <site name="R_palm" size="0.008" pos="-0.035 -0.09 0" quat="0 0.707107 -0.707107 0" group="4" rgba="0 1 0 1"/>
                            <camera name="R_wrist_cam" quat="0.48481 -0.87462 0 0" pos="-0.035 0.01 -0.09" fovy="90"/>
                            <body name="R_thumb_proximal_base" pos="-0.01696 -0.0691 -0.02045" quat="0.499998 0.5 0.500002 -0.5" gravcomp="1">
                              <inertial pos="-0.0048064 0.0009382 -0.00757" quat="0.515015 0.680854 0.408023 0.323596" mass="0.0018869" diaginertia="8.66026e-08 6.8732e-08 4.94194e-08"/>
                              <joint name="R_thumb_proximal_yaw_joint" range="0 1.308" armature="0.002" damping="0.05" frictionloss="0.001" axis="0 0 -1"/>
                              <geom name="geom33" pos="0.00112392 -0.00175793 -0.00842112" quat="0.671503 0.725909 0.107145 0.103246" type="mesh" contype="2" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom33_hull"/>
                              <body name="R_thumb_proximal" pos="0.0099867 -0.0098242 -0.0089" quat="0.704571 0.704573 -0.0598167 -0.0598169" gravcomp="1">
                                <inertial pos="0.021932 0.012785 -0.0080386" quat="-0.254474 0.660716 0.251893 0.659733" mass="0.0066075" diaginertia="2.78601e-06 2.43933e-06 8.64566e-07"/>
                                <joint name="R_thumb_proximal_pitch_joint" range="0 0.6" armature="0.002" damping="0.05" frictionloss="0.001"/>
                                <geom name="geom35" pos="0.0245196 0.0148876 -0.00799809" quat="-0.199905 0.67841 0.199478 0.678238" type="mesh" contype="2" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom35_hull"/>
                                <body name="R_thumb_intermediate" pos="0.04407 0.034553 -0.0008" gravcomp="1">
                                  <inertial pos="0.0095544 -0.0016282 -0.0071997" quat="0.636718 0.307389 -0.636802 0.307548" mass="0.0037847" diaginertia="4.6531e-07 4.48089e-07 2.45661e-07"/>
                                  <joint name="R_thumb_intermediate_joint" range="0 0.8" armature="0.002" damping="0.05" frictionloss="0.001"/>
                                  <geom name="geom37" pos="0.00668695 -0.00197589 -0.00719975" quat="-0.218212 0.672629 0.218137 0.672585" type="mesh" contype="2" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom37_hull"/>
                                  <body name="R_thumb_distal" pos="0.020248 0.010156 -0.0012" gravcomp="1">
                                    <inertial pos="0.0092888 0.0049529 -0.0060033" quat="-0.266294 0.655967 0.262806 0.655537" mass="0.0033441" diaginertia="2.0026e-07 1.95247e-07 8.1593e-08"/>
                                    <joint name="R_thumb_distal_joint" range="0 0.4" armature="0.002" damping="0.05" frictionloss="0.001"/>
                                    <geom name="geom39" pos="0.000567032 0.000252357 -0.00600003" quat="0.70562 0.705422 -0.0473326 -0.0473342" type="mesh" contype="2" conaffinity="5" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom39_hull"/>
                                    <site name="R_thumb_tip" size="0.006" pos="0.0202 0.014 -0.006" group="4" rgba="1 0.5 0 1"/>
                                  </body>
                                </body>
                              </body>
                            </body>
                            <body name="R_index_proximal" pos="0.00028533 -0.13653 -0.032268" quat="0.0174454 -0.999848 0 0" gravcomp="1">
                              <inertial pos="0.0012259 0.011942 -0.0060001" quat="0.50867 0.49121 -0.508643 0.491172" mass="0.0042403" diaginertia="6.9398e-07 6.62871e-07 2.10909e-07"/>
                              <joint name="R_index_proximal_joint" range="0 1.47" armature="0.002" damping="0.05" frictionloss="0.001"/>
                              <geom name="geom41" pos="0.000623522 0.0123596 -0.00269918" quat="0.519896 0.510346 -0.485531 0.483234" type="mesh" contype="2" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom41_hull"/>
                              <body name="R_index_intermediate" pos="-0.0026138 0.032026 -0.001" gravcomp="1">
                                <inertial pos="0.0019697 0.019589 -0.005" quat="0.466766 0.531146 -0.46678 0.531159" mass="0.0045683" diaginertia="7.8179e-07 7.72465e-07 8.47212e-08"/>
                                <joint name="R_index_intermediate_joint" range="-0.04545 1.56" armature="0.002" damping="0.05" frictionloss="0.001"/>
                                <geom name="geom43" pos="0.000591064 0.030926 -0.005" quat="0.701352 0.701362 0.0899491 0.0900313" type="mesh" contype="2" conaffinity="5" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom43_hull"/>
                                <site name="R_index_tip" size="0.006" pos="-0.0008 0.045 -0.005" group="4" rgba="1 0.5 0 1"/>
                              </body>
                            </body>
                            <body name="R_middle_proximal" pos="0.00028533 -0.1371 -0.01295" quat="1.32679e-06 1 0 0" gravcomp="1">
                              <inertial pos="0.001297 0.011934 -0.0060001" quat="0.510131 0.489693 -0.510105 0.489653" mass="0.0042403" diaginertia="6.9397e-07 6.62865e-07 2.10915e-07"/>
                              <joint name="R_middle_proximal_joint" range="0 1.47" armature="0.002" damping="0.05" frictionloss="0.001"/>
                              <geom name="geom45" pos="0.000623522 0.0123596 -0.00269918" quat="0.519896 0.510346 -0.485531 0.483234" type="mesh" contype="2" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom45_hull"/>
                              <body name="R_middle_intermediate" pos="-0.0024229 0.032041 -0.001" gravcomp="1">
                                <inertial pos="0.001921 0.020796 -0.005" quat="0.466148 0.531627 -0.466229 0.531705" mass="0.0050396" diaginertia="9.8384e-07 9.73279e-07 9.14014e-08"/>
                                <joint name="R_middle_intermediate_joint" range="-0.04545 1.56" armature="0.002" damping="0.05" frictionloss="0.001"/>
                                <geom name="geom47" pos="0.00168336 0.0185577 -0.00499995" quat="0.469804 0.528047 -0.470283 0.528474" type="mesh" contype="2" conaffinity="5" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom47_hull"/>
                                <site name="R_middle_tip" size="0.006" pos="-0.001 0.048 -0.005" group="4" rgba="1 0.5 0 1"/>
                              </body>
                            </body>
                            <body name="R_ring_proximal" pos="0.00028533 -0.13691 0.0062872" quat="0.0261933 0.999657 0 0" gravcomp="1">
                              <inertial pos="0.001297 0.011934 -0.0060002" quat="0.510129 0.489691 -0.510107 0.489654" mass="0.0042403" diaginertia="6.9397e-07 6.62865e-07 2.10915e-07"/>
                              <joint name="R_ring_proximal_joint" range="0 1.47" armature="0.002" damping="0.05" frictionloss="0.001"/>
                              <geom name="geom49" pos="0.000623522 0.0123596 -0.00269918" quat="0.519896 0.510346 -0.485531 0.483234" type="mesh" contype="2" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom49_hull"/>
                              <body name="R_ring_intermediate" pos="-0.0024229 0.032041 -0.001" gravcomp="1">
                                <inertial pos="0.0021753 0.019567 -0.005" quat="0.469539 0.528681 -0.46957 0.528709" mass="0.0045683" diaginertia="7.8177e-07 7.72448e-07 8.4722e-08"/>
                                <joint name="R_ring_intermediate_joint" range="-0.04545 1.56" armature="0.002" damping="0.05" frictionloss="0.001"/>
                                <geom name="geom51" pos="0.000591064 0.030926 -0.005" quat="0.701352 0.701362 0.0899491 0.0900313" type="mesh" contype="2" conaffinity="5" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom51_hull"/>
                                <site name="R_ring_tip" size="0.006" pos="-0.0008 0.045 -0.005" group="4" rgba="1 0.5 0 1"/>
                              </body>
                            </body>
                            <body name="R_pinky_proximal" pos="0.00028533 -0.13571 0.025488" quat="0.0523224 0.99863 0 0" gravcomp="1">
                              <inertial pos="0.001297 0.011934 -0.0060001" quat="0.51013 0.489693 -0.510106 0.489653" mass="0.0042403" diaginertia="6.9397e-07 6.62865e-07 2.10915e-07"/>
                              <joint name="R_pinky_proximal_joint" range="0 1.47" armature="0.002" damping="0.05" frictionloss="0.001"/>
                              <geom name="geom53" pos="0.000623522 0.0123596 -0.00269918" quat="0.519896 0.510346 -0.485531 0.483234" type="mesh" contype="2" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom53_hull"/>
                              <body name="R_pinky_intermediate" pos="-0.0024229 0.032041 -0.001" gravcomp="1">
                                <inertial pos="0.0024748 0.016203 -0.0050031" quat="0.47398 0.528862 -0.469291 0.524799" mass="0.0035996" diaginertia="4.4867e-07 4.43723e-07 6.56538e-08"/>
                                <joint name="R_pinky_intermediate_joint" range="-0.04545 1.56" armature="0.002" damping="0.05" frictionloss="0.001"/>
                                <geom name="geom55" pos="0.00237228 0.0137586 -0.00499999" quat="0.473377 0.525304 -0.473345 0.525275" type="mesh" contype="2" conaffinity="5" condim="4" group="3" priority="1" friction="1.5 0.02 0.002" solref="0.004" solimp="0.95 0.99" rgba="0.8 0.3 0.3 1" mesh="geom55_hull"/>
                                <site name="R_pinky_tip" size="0.006" pos="-0.0008 0.037 -0.005" group="4" rgba="1 0.5 0 1"/>
                              </body>
                            </body>
                          </body>
                        </body>
                      </body>
                    </body>
                  </body>
                </body>
              </body>
            </body>
          </body>
        </body>
      </body>
    </body>
    <body name="box" pos="0.33 -0.18 0.814833">
      <joint name="box_free" type="free"/>
      <geom name="box" size="0.0360367 0.0441613 0.013833" pos="0.000349456 -0.000517507 -0.000911" quat="0.993323 0 0 0.115371" type="box" condim="4" friction="1 0.01 0.001" solref="0.004" rgba="0.9 0.3 0.4 1" mass="0.1"/>
    </body>
  </worldbody>

  <contact>
    <exclude body1="torso_link" body2="right_shoulder_roll_link"/>
    <exclude body1="right_wrist_roll_link" body2="R_hand_base_link"/>
    <exclude body1="right_wrist_pitch_link" body2="R_hand_base_link"/>
    <exclude body1="right_wrist_yaw_link" body2="R_hand_base_link"/>
  </contact>

  <equality>
    <joint joint1="R_thumb_intermediate_joint" joint2="R_thumb_proximal_pitch_joint" polycoef="0 1.334 0 0 0" solref="0.005"/>
    <joint joint1="R_thumb_distal_joint" joint2="R_thumb_proximal_pitch_joint" polycoef="0 0.667 0 0 0" solref="0.005"/>
    <joint joint1="R_index_intermediate_joint" joint2="R_index_proximal_joint" polycoef="-0.04545 1.06399 0 0 0" solref="0.005"/>
    <joint joint1="R_middle_intermediate_joint" joint2="R_middle_proximal_joint" polycoef="-0.04545 1.06399 0 0 0" solref="0.005"/>
    <joint joint1="R_ring_intermediate_joint" joint2="R_ring_proximal_joint" polycoef="-0.04545 1.06399 0 0 0" solref="0.005"/>
    <joint joint1="R_pinky_intermediate_joint" joint2="R_pinky_proximal_joint" polycoef="-0.04545 1.06399 0 0 0" solref="0.005"/>
  </equality>

  <actuator>
    <general name="right_shoulder_pitch_joint" joint="right_shoulder_pitch_joint" ctrlrange="-3.0892 2.6704" biastype="affine" gainprm="400" biasprm="0 -400 1"/>
    <general name="right_shoulder_roll_joint" joint="right_shoulder_roll_joint" ctrlrange="-2.2515 1.5882" biastype="affine" gainprm="400" biasprm="0 -400 1"/>
    <general name="right_shoulder_yaw_joint" joint="right_shoulder_yaw_joint" ctrlrange="-2.618 2.618" biastype="affine" gainprm="400" biasprm="0 -400 1"/>
    <general name="right_elbow_joint" joint="right_elbow_joint" ctrlrange="-1.0472 2.0944" biastype="affine" gainprm="400" biasprm="0 -400 1"/>
    <general name="right_wrist_roll_joint" joint="right_wrist_roll_joint" ctrlrange="-1.97222 1.97222" biastype="affine" gainprm="400" biasprm="0 -400 1"/>
    <general name="right_wrist_pitch_joint" joint="right_wrist_pitch_joint" ctrlrange="-1.61443 1.61443" biastype="affine" gainprm="400" biasprm="0 -400 1"/>
    <general name="right_wrist_yaw_joint" joint="right_wrist_yaw_joint" ctrlrange="-1.61443 1.61443" biastype="affine" gainprm="400" biasprm="0 -400 1"/>
    <general name="R_thumb_proximal_yaw_joint" joint="R_thumb_proximal_yaw_joint" ctrlrange="0 1.308" forcerange="-0.8 0.8" biastype="affine" gainprm="20" biasprm="0 -20 1"/>
    <general name="R_thumb_proximal_pitch_joint" joint="R_thumb_proximal_pitch_joint" ctrlrange="0 0.6" forcerange="-0.8 0.8" biastype="affine" gainprm="20" biasprm="0 -20 1"/>
    <general name="R_index_proximal_joint" joint="R_index_proximal_joint" ctrlrange="0 1.47" forcerange="-0.8 0.8" biastype="affine" gainprm="20" biasprm="0 -20 1"/>
    <general name="R_middle_proximal_joint" joint="R_middle_proximal_joint" ctrlrange="0 1.47" forcerange="-0.8 0.8" biastype="affine" gainprm="20" biasprm="0 -20 1"/>
    <general name="R_ring_proximal_joint" joint="R_ring_proximal_joint" ctrlrange="0 1.47" forcerange="-0.8 0.8" biastype="affine" gainprm="20" biasprm="0 -20 1"/>
    <general name="R_pinky_proximal_joint" joint="R_pinky_proximal_joint" ctrlrange="0 1.47" forcerange="-0.8 0.8" biastype="affine" gainprm="20" biasprm="0 -20 1"/>
  </actuator>
</mujoco>

In [ ]:
%%writefile flat_grasp.json
{
  "palm_pos": [
    0.2931296577395578,
    -0.195,
    0.9502218378509392
  ],
  "palm_quat": [
    0.62721137512625,
    -0.62721137512625,
    0.3265055756219769,
    0.3265055756219769
  ],
  "arm_q": [
    -0.7943590236601599,
    -1.1467569765504113,
    0.7386906025253164,
    -0.3906639626371191,
    -0.3363642074983204,
    0.8560757389576806,
    1.1429469855912895
  ],
  "box_xy": [
    0.33,
    -0.18
  ],
  "box_half": [
    0.0446225015891141,
    0.05044700134482964,
    0.013832999734781465
  ]
}

Throughput (policy steps / s on one GPU), a learning check (20M steps), then the real run.

In [ ]:
!python mjx_bench.py flat_grasp_train.xml --envs 1024 2>&1 | grep -v Warning

In [ ]:
!python train_flat_grasp.py --steps 20000000 --evals 6 --out check 2>&1 | grep -v Warning

In [ ]:
!python train_flat_grasp.py --steps 150000000 --evals 30 --out out 2>&1 | grep -v Warning

In [ ]:
!cd out && zip -q -r ../flat_grasp_out.zip . && ls -la /kaggle/working/out